# TESSCut Conditional Detrending Notebook

This notebook demonstrates the detrending workflow developed after the light-curve trend/flattening checks.

**Goal:** detrend only the light curves where detrending is scientifically justified.

Main design decision:

```text
Only consider TESSCut light curves for detrending.
Detect robust segment-wise drift first.
Apply detrending only when drift is detected.
Do not detrend SPOC or QLP rows in this workflow.
```

Why this conservative approach?

- TESSCut light curves are more likely to contain aperture/background/systematic drift.
- SPOC and QLP products are already pipeline processed.
- Some variable-star families have real long-timescale astrophysical variability, so blindly flattening every light curve can remove real signal.

## 1. Configuration

Set the input parquet file, the base folder containing FITS files, and the output folder.

The notebook reads `rawLightCurvePath` from the parquet. If the path is relative, it is resolved against `fitsFileFolder`. If it is absolute and exists, it is used directly.

In [ ]:
from pathlib import Path
import os
import shutil
import warnings
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

from astropy.io import fits

# ----------------------------
# User configuration
# ----------------------------

inputParquetFile = Path("/data/projects/TESS-research/data_pipeline/TESSCache/TESSAugmented_QC.parquet")

# Base folder where raw FITS files are stored.
# Example: /data/projects/TESS-research/data_pipeline/TESSCache
fitsFileFolder = Path("/data/projects/TESS-research/data_pipeline/TESSCache")

# Output folder for detrended FITS files and updated parquet.
outputFolder = Path("/data/projects/TESS-research/detrend_tesscut/output")
outputFolder.mkdir(parents=True, exist_ok=True)

# Main path column from metadata.
rawPathColumn = "rawLightCurvePath"

# Provenance column.
provenanceColumn = "provenance"

# Only these provenance values are eligible for detrending.
eligibleProvenance = {"TESSCut"}

# ----------------------------
# Robust drift detection parameters
# ----------------------------

gapThresholdDays = 1.0
minFinitePoints = 100
minSegmentPoints = 100
minSegmentDurationDays = 5.0

# Minimal-fix threshold from our trend-analysis discussion.
segmentDriftStrengthThreshold = 7.0

# Final star-level flag:
# drift if at least 50% of valid segments have drift strength >= threshold.
fractionSegmentsWithDriftThreshold = 0.5

detrendMethod = "tesscut_segment_linear_mad_conditional_v1"

## Drift Detection Threshold Selection

### Segment Drift Strength Threshold = 7.0

For each contiguous observing segment, we define the drift strength as:

\[
\text{segmentDriftStrength} =
\frac{|\text{slope} \times \text{segmentDuration}|}
{\text{robustScatter}}
\]

where:

- `slope × segmentDuration` estimates the total fitted drift across the segment
- `robustScatter` is estimated using the Median Absolute Deviation (MAD)

This metric measures how large the long-term drift is relative to the typical variability or noise level within the segment.

A threshold of `7.0` means that the fitted drift across the segment must be substantially larger than the robust scatter before the segment is considered strongly drifting.

Lower thresholds (e.g. 2–3) produced unrealistically high drift rates (>80–90%) because many variable stars naturally exhibit mild slopes, curvature, incomplete cycles, or low-frequency variability. Increasing the threshold significantly reduced false positives caused by:

- intrinsic stellar variability
- incomplete periodic cycles
- observational noise
- sector stitching artifacts
- TESSCut background/aperture systematics

Therefore, `7.0` was selected empirically as a conservative threshold that identifies only strong and dominant drift behavior.

---

### Fraction of Segments With Drift Threshold = 0.5

A stitched light curve may contain multiple observing segments separated by large temporal gaps.

A light curve is classified as drifting only if:

\[
\text{fractionSegmentsWithDrift} \ge 0.5
\]

This means that at least 50% of valid observing segments must independently exhibit strong drift behavior.

This criterion was introduced because isolated problematic segments may occur due to:

- instrumental artifacts
- aperture contamination
- background estimation errors
- sector-level calibration offsets
- localized observational issues

Using a majority-based threshold helps ensure that detected drift is persistent across the light curve rather than driven by a single anomalous segment.

The choice of `0.5` provides a simple majority criterion that improves robustness while avoiding over-sensitivity to isolated failures.

## 2. Helper functions

The helper functions below follow the logic from the trend-analysis work:

1. Read `TIME` and `FLUX` directly from pipeline-generated raw FITS using `astropy.io.fits`.
2. Remove only non-finite values for detection.
3. Split light curves into contiguous observing segments using time gaps.
4. For each valid segment, fit a linear drift model.
5. Compute drift strength:

```text
segmentDriftStrength = abs(slope × segmentDurationDays) / (1.4826 × MAD)
```

6. Flag a light curve as drifting only if at least 50% of valid segments exceed the threshold.

In [ ]:
def resolve_raw_fits_path(path_value, fits_base: Path) -> Optional[Path]:
    """
    Resolve raw FITS path from metadata.

    Rules:
    1. If path is absolute and exists, use it.
    2. If path is relative to fits_base and exists, use fits_base / path.
    3. If only filename works under fits_base, use fits_base / basename.
    """
    if pd.isna(path_value):
        return None

    p = Path(str(path_value))

    if p.is_absolute() and p.exists():
        return p

    candidate = fits_base / p
    if candidate.exists():
        return candidate

    candidate = fits_base / p.name
    if candidate.exists():
        return candidate

    return None


def load_time_flux_from_raw_fits(raw_fits_path: Path) -> Tuple[np.ndarray, np.ndarray, fits.HDUList]:
    """
    Load TIME and FLUX from HDU[1] of a pipeline-generated FITS file.

    Returns time, flux, and the opened HDUList. Caller should close HDUList.
    """
    hdul = fits.open(raw_fits_path, memmap=False)

    if len(hdul) <= 1 or hdul[1].data is None:
        hdul.close()
        raise ValueError("Missing table data in HDU[1]")

    table = hdul[1].data
    names = set(getattr(table, "names", []) or [])

    if "TIME" not in names or "FLUX" not in names:
        hdul.close()
        raise ValueError("HDU[1] must contain TIME and FLUX columns")

    time = np.asarray(table["TIME"], dtype=float)
    flux = np.asarray(table["FLUX"], dtype=float)

    return time, flux, hdul


def split_into_segments(time: np.ndarray, flux: np.ndarray, gap_days: float) -> List[Tuple[np.ndarray, np.ndarray, np.ndarray]]:
    """
    Split sorted finite time/flux arrays into contiguous segments.

    Returns list of (segment_indices, segment_time, segment_flux).
    segment_indices refer to positions in the sorted finite arrays.
    """
    if len(time) == 0:
        return []

    dt = np.diff(time)
    breaks = np.where(dt > gap_days)[0] + 1
    starts = np.concatenate(([0], breaks))
    ends = np.concatenate((breaks, [len(time)]))

    segments = []
    for start, end in zip(starts, ends):
        if end > start:
            idx = np.arange(start, end)
            segments.append((idx, time[start:end], flux[start:end]))

    return segments


def analyze_segment_drift(
    segment_time: np.ndarray,
    segment_flux: np.ndarray,
    threshold: float,
    min_points: int,
    min_duration_days: float,
) -> Optional[Dict]:
    """
    Analyze linear drift in one contiguous segment.

    The metric compares total fitted linear drift across the segment
    to robust scatter estimated by 1.4826*MAD.
    """
    n = len(segment_time)
    if n < min_points:
        return None

    duration = float(segment_time[-1] - segment_time[0])
    if not np.isfinite(duration) or duration < min_duration_days:
        return None

    median_flux = float(np.median(segment_flux))
    mad = float(np.median(np.abs(segment_flux - median_flux)))
    robust_scatter = 1.4826 * mad

    if not np.isfinite(robust_scatter) or robust_scatter <= 0:
        return None

    t_centered = segment_time - np.median(segment_time)

    try:
        slope, intercept = np.polyfit(t_centered, segment_flux, 1)
    except Exception:
        return None

    drift = abs(float(slope) * duration)
    drift_strength = drift / robust_scatter

    if not np.isfinite(drift_strength):
        return None

    return {
        "segmentNumPoints": int(n),
        "segmentDurationDays": float(duration),
        "segmentSlope": float(slope),
        "segmentIntercept": float(intercept),
        "segmentDrift": float(drift),
        "segmentRobustScatter": float(robust_scatter),
        "segmentDriftStrength": float(drift_strength),
        "segmentHasDrift": bool(drift_strength >= threshold),
    }


def detect_robust_drift_for_file(raw_fits_path: Path) -> Dict:
    """
    Run segment-wise robust drift detection for one raw FITS file.
    """
    try:
        time, flux, hdul = load_time_flux_from_raw_fits(raw_fits_path)
        hdul.close()

        finite_mask = np.isfinite(time) & np.isfinite(flux)
        time_finite = time[finite_mask]
        flux_finite = flux[finite_mask]

        if len(time_finite) < minFinitePoints:
            return {
                "trendStatus": "insufficient_data",
                "trendDetected": False,
                "trendNumFinitePoints": int(len(time_finite)),
                "validSegmentCount": 0,
                "segmentDriftCount": 0,
                "fractionSegmentsWithDrift": np.nan,
                "maxSegmentDriftStrength": np.nan,
                "medianSegmentDriftStrength": np.nan,
            }

        sort_idx = np.argsort(time_finite)
        time_finite = time_finite[sort_idx]
        flux_finite = flux_finite[sort_idx]

        segments = split_into_segments(time_finite, flux_finite, gapThresholdDays)

        valid_results = []
        for _idx, seg_time, seg_flux in segments:
            result = analyze_segment_drift(
                segment_time=seg_time,
                segment_flux=seg_flux,
                threshold=segmentDriftStrengthThreshold,
                min_points=minSegmentPoints,
                min_duration_days=minSegmentDurationDays,
            )
            if result is not None:
                valid_results.append(result)

        if len(valid_results) == 0:
            return {
                "trendStatus": "insufficient_data",
                "trendDetected": False,
                "trendNumFinitePoints": int(len(time_finite)),
                "validSegmentCount": 0,
                "segmentDriftCount": 0,
                "fractionSegmentsWithDrift": np.nan,
                "maxSegmentDriftStrength": np.nan,
                "medianSegmentDriftStrength": np.nan,
            }

        strengths = np.array([r["segmentDriftStrength"] for r in valid_results], dtype=float)
        drift_count = int(sum(r["segmentHasDrift"] for r in valid_results))
        valid_count = len(valid_results)
        frac = drift_count / valid_count

        detected = bool(frac >= fractionSegmentsWithDriftThreshold)

        return {
            "trendStatus": "drift" if detected else "no_drift",
            "trendDetected": detected,
            "trendNumFinitePoints": int(len(time_finite)),
            "validSegmentCount": int(valid_count),
            "segmentDriftCount": int(drift_count),
            "fractionSegmentsWithDrift": float(frac),
            "maxSegmentDriftStrength": float(np.max(strengths)),
            "medianSegmentDriftStrength": float(np.median(strengths)),
        }

    except Exception as e:
        return {
            "trendStatus": "failed",
            "trendDetected": False,
            "trendError": str(e),
            "trendNumFinitePoints": 0,
            "validSegmentCount": 0,
            "segmentDriftCount": 0,
            "fractionSegmentsWithDrift": np.nan,
            "maxSegmentDriftStrength": np.nan,
            "medianSegmentDriftStrength": np.nan,
        }

## 3. Detrending method

For a TESSCut light curve that is flagged as drifting, this notebook applies **segment-wise linear detrending**.

For each valid segment:

```text
flux_detrended = flux - fitted_linear_model + segment_median_flux
```

This removes slow linear drift while preserving the segment's median flux scale.

Important safeguards:

- Only TESSCut rows are eligible.
- Only rows with detected robust drift are detrended.
- The original raw FITS files are not overwritten.
- SPOC and QLP rows are kept unchanged and marked `detrended = False`.

In [ ]:
def build_detrended_filename(raw_fits_path: Path) -> str:
    """
    Convert xxxx_raw.fits to xxxx_detrended.fits.
    """
    name = raw_fits_path.name
    if name.endswith("_raw.fits"):
        return name.replace("_raw.fits", "_detrended.fits")
    if name.endswith(".fits"):
        return name.replace(".fits", "_detrended.fits")
    return name + "_detrended.fits"


def detrend_raw_fits_file(raw_fits_path: Path, output_folder: Path) -> Dict:
    """
    Create a detrended FITS file using segment-wise linear detrending.

    Non-finite rows are preserved in position; only finite rows are used
    for fitting and corrected where possible.
    """
    hdul = None
    try:
        time, flux, hdul = load_time_flux_from_raw_fits(raw_fits_path)

        finite_mask = np.isfinite(time) & np.isfinite(flux)
        finite_positions = np.where(finite_mask)[0]

        if len(finite_positions) < minFinitePoints:
            hdul.close()
            return {
                "detrendStatus": "insufficient_data",
                "detrended": False,
                "detrendLightCurvePath": None,
                "detrendError": None,
                "detrendSegmentCount": 0,
                "detrendCorrectedSegmentCount": 0,
            }

        # Work in sorted finite space, then map back to original row positions.
        time_finite = time[finite_mask]
        flux_finite = flux[finite_mask]
        sort_idx = np.argsort(time_finite)

        time_sorted = time_finite[sort_idx]
        flux_sorted = flux_finite[sort_idx]
        original_positions_sorted = finite_positions[sort_idx]

        detrended_flux = np.array(flux, dtype=float, copy=True)

        segments = split_into_segments(time_sorted, flux_sorted, gapThresholdDays)

        corrected_segments = 0
        valid_segments = 0

        for seg_sorted_idx, seg_time, seg_flux in segments:
            analysis = analyze_segment_drift(
                segment_time=seg_time,
                segment_flux=seg_flux,
                threshold=segmentDriftStrengthThreshold,
                min_points=minSegmentPoints,
                min_duration_days=minSegmentDurationDays,
            )

            if analysis is None:
                continue

            valid_segments += 1

            # Detrend every valid segment for a file already selected for detrending.
            t_centered = seg_time - np.median(seg_time)
            slope, intercept = np.polyfit(t_centered, seg_flux, 1)
            model = slope * t_centered + intercept
            segment_median = np.median(seg_flux)
            corrected_flux = seg_flux - model + segment_median

            original_positions = original_positions_sorted[seg_sorted_idx]
            detrended_flux[original_positions] = corrected_flux
            corrected_segments += 1

        if corrected_segments == 0:
            hdul.close()
            return {
                "detrendStatus": "no_valid_segments",
                "detrended": False,
                "detrendLightCurvePath": None,
                "detrendError": None,
                "detrendSegmentCount": int(valid_segments),
                "detrendCorrectedSegmentCount": 0,
            }

        # Create output HDUList by copying original structure.
        new_hdul = fits.HDUList([hdu.copy() for hdu in hdul])
        hdul.close()

        table_hdu = new_hdul[1]
        table_data = table_hdu.data

        # Rebuild table with detrended FLUX and optional ORIGINAL_FLUX.
        rebuilt_cols = []
        existing_names = list(table_data.names)

        for col in table_hdu.columns:
            if col.name == "FLUX":
                rebuilt_cols.append(
                    fits.Column(
                        name="FLUX",
                        array=np.asarray(detrended_flux, dtype=np.float32),
                        format=col.format,
                    )
                )
            else:
                rebuilt_cols.append(
                    fits.Column(
                        name=col.name,
                        array=table_data[col.name],
                        format=col.format,
                    )
                )

        if "ORIGINAL_FLUX" not in existing_names:
            rebuilt_cols.append(
                fits.Column(
                    name="ORIGINAL_FLUX",
                    array=np.asarray(flux, dtype=np.float32),
                    format="E",
                )
            )

        new_table_hdu = fits.BinTableHDU.from_columns(rebuilt_cols)
        new_table_hdu.header.extend(table_hdu.header, update=True, strip=True)

        # Record detrending metadata.
        new_table_hdu.header["DETRND"] = (True, "Segment-wise linear detrending applied")
        new_table_hdu.header["DTRMETH"] = (detrendMethod, "Detrending method")
        new_table_hdu.header["DTRTHR"] = (float(segmentDriftStrengthThreshold), "Drift-strength threshold")
        new_table_hdu.header["DTRFRAC"] = (float(fractionSegmentsWithDriftThreshold), "Segment fraction threshold")
        new_table_hdu.header["GAPDAYS"] = (float(gapThresholdDays), "Gap threshold in days")

        new_hdul[1] = new_table_hdu

        output_name = build_detrended_filename(raw_fits_path)
        output_path = output_folder / output_name
        new_hdul.writeto(output_path, overwrite=True)
        new_hdul.close()

        return {
            "detrendStatus": "detrended",
            "detrended": True,
            "detrendLightCurvePath": str(output_path),
            "detrendError": None,
            "detrendSegmentCount": int(valid_segments),
            "detrendCorrectedSegmentCount": int(corrected_segments),
        }

    except Exception as e:
        if hdul is not None:
            try:
                hdul.close()
            except Exception:
                pass

        return {
            "detrendStatus": "failed",
            "detrended": False,
            "detrendLightCurvePath": None,
            "detrendError": str(e),
            "detrendSegmentCount": 0,
            "detrendCorrectedSegmentCount": 0,
        }

## 4. Load metadata and filter TESSCut rows

The notebook processes all rows for metadata consistency, but detrending is only attempted for rows with:

```text
provenance == "TESSCut"
```

In [ ]:
df = pd.read_parquet(inputParquetFile)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Input parquet:", inputParquetFile)

if rawPathColumn not in df.columns:
    raise ValueError(f"Required column missing: {rawPathColumn}")

if provenanceColumn not in df.columns:
    raise ValueError(f"Required column missing: {provenanceColumn}")

display(df[provenanceColumn].value_counts(dropna=False).to_frame("count"))

tesscut_mask = df[provenanceColumn].isin(eligibleProvenance)
print("TESSCut rows:", int(tesscut_mask.sum()))
print("Non-TESSCut rows:", int((~tesscut_mask).sum()))

## 5. Run TESSCut trend detection and conditional detrending

This cell:

1. Resolves the raw FITS path.
2. Runs robust drift detection for TESSCut rows.
3. Detrends only TESSCut rows with detected drift.
4. Marks all SPOC/QLP/non-TESSCut rows as `detrended = False`.
5. Stores trend and detrending statistics back into the metadata dataframe.

In [ ]:
# New columns to be added/updated.
new_columns_defaults = {
    "trendDetectionEligible": False,
    "trendDetectedForDetrend": False,
    "trendDetectionStatus": None,
    "trendDetectionMethod": "segment_wise_linear_drift_mad_v1",
    "trendDetectionThreshold": segmentDriftStrengthThreshold,
    "trendFractionSegmentsWithDrift": np.nan,
    "trendValidSegmentCount": 0,
    "trendSegmentDriftCount": 0,
    "trendMaxSegmentDriftStrength": np.nan,
    "trendMedianSegmentDriftStrength": np.nan,
    "trendNumFinitePoints": 0,
    "detrended": False,
    "detrendEligible": False,
    "detrendStatus": "not_eligible",
    "detrendReason": None,
    "detrendMethod": None,
    "detrendLightCurvePath": None,
    "detrendInputRawLightCurvePath": None,
    "detrendResolvedRawFitsPath": None,
    "detrendSegmentCount": 0,
    "detrendCorrectedSegmentCount": 0,
    "detrendError": None,
}

for col, default in new_columns_defaults.items():
    if col not in df.columns:
        df[col] = default

processed = 0
missing_raw = 0
detected_count = 0
detrended_count = 0
failed_count = 0

for idx, row in df.iterrows():
    provenance = row[provenanceColumn]

    # Default for non-TESSCut rows.
    if provenance not in eligibleProvenance:
        df.at[idx, "trendDetectionEligible"] = False
        df.at[idx, "detrendEligible"] = False
        df.at[idx, "detrended"] = False
        df.at[idx, "detrendStatus"] = "not_eligible"
        df.at[idx, "detrendReason"] = f"provenance={provenance}; only TESSCut eligible"
        continue

    df.at[idx, "trendDetectionEligible"] = True
    df.at[idx, "detrendEligible"] = True
    df.at[idx, "detrendInputRawLightCurvePath"] = row[rawPathColumn]

    raw_fits_path = resolve_raw_fits_path(row[rawPathColumn], fitsFileFolder)

    if raw_fits_path is None:
        missing_raw += 1
        df.at[idx, "trendDetectionStatus"] = "missing_raw_fits"
        df.at[idx, "trendDetectedForDetrend"] = False
        df.at[idx, "detrended"] = False
        df.at[idx, "detrendStatus"] = "missing_raw_fits"
        df.at[idx, "detrendReason"] = "raw FITS path could not be resolved"
        continue

    df.at[idx, "detrendResolvedRawFitsPath"] = str(raw_fits_path)

    trend = detect_robust_drift_for_file(raw_fits_path)

    df.at[idx, "trendDetectionStatus"] = trend.get("trendStatus")
    df.at[idx, "trendDetectedForDetrend"] = bool(trend.get("trendDetected", False))
    df.at[idx, "trendFractionSegmentsWithDrift"] = trend.get("fractionSegmentsWithDrift", np.nan)
    df.at[idx, "trendValidSegmentCount"] = trend.get("validSegmentCount", 0)
    df.at[idx, "trendSegmentDriftCount"] = trend.get("segmentDriftCount", 0)
    df.at[idx, "trendMaxSegmentDriftStrength"] = trend.get("maxSegmentDriftStrength", np.nan)
    df.at[idx, "trendMedianSegmentDriftStrength"] = trend.get("medianSegmentDriftStrength", np.nan)
    df.at[idx, "trendNumFinitePoints"] = trend.get("trendNumFinitePoints", 0)

    if "trendError" in trend:
        df.at[idx, "detrendError"] = trend["trendError"]

    if not bool(trend.get("trendDetected", False)):
        df.at[idx, "detrended"] = False
        df.at[idx, "detrendStatus"] = "not_detrended_no_drift"
        df.at[idx, "detrendReason"] = "TESSCut but robust drift not detected"
        processed += 1
        continue

    detected_count += 1

    detrend_result = detrend_raw_fits_file(raw_fits_path, outputFolder)

    for key, value in detrend_result.items():
        if key in df.columns:
            df.at[idx, key] = value

    df.at[idx, "detrendMethod"] = detrendMethod
    df.at[idx, "detrendReason"] = "TESSCut and robust drift detected"

    if detrend_result.get("detrended", False):
        detrended_count += 1
    elif detrend_result.get("detrendStatus") == "failed":
        failed_count += 1

    processed += 1

    if processed % 250 == 0:
        print(
            f"Processed TESSCut rows: {processed} | "
            f"trend detected: {detected_count} | detrended: {detrended_count} | "
            f"missing raw: {missing_raw} | failed: {failed_count}"
        )

print("Done.")
print("TESSCut processed:", processed)
print("TESSCut trend detected:", detected_count)
print("TESSCut detrended:", detrended_count)
print("Missing raw FITS:", missing_raw)
print("Detrend failed:", failed_count)

## 6. Summary statistics

These summaries are important for deciding whether the results are plausible.

Expected pattern:

- `SPOC` and `QLP` should have `detrended = False`.
- Only `TESSCut` rows should be eligible.
- A subset of TESSCut rows should be detrended, not all TESSCut rows.

In [ ]:
summary_by_provenance = (
    df.groupby(provenanceColumn, dropna=False)
      .agg(
          total=(provenanceColumn, "size"),
          eligible=("detrendEligible", "sum"),
          trend_detected=("trendDetectedForDetrend", "sum"),
          detrended=("detrended", "sum"),
          median_fraction_segments_with_drift=("trendFractionSegmentsWithDrift", "median"),
          median_drift_strength=("trendMedianSegmentDriftStrength", "median"),
          p75_drift_strength=("trendMedianSegmentDriftStrength", lambda x: x.dropna().quantile(0.75) if len(x.dropna()) else np.nan),
          failed=("detrendStatus", lambda x: (x == "failed").sum()),
          missing_raw=("detrendStatus", lambda x: (x == "missing_raw_fits").sum()),
      )
      .reset_index()
)

summary_by_provenance["trend_detected_percent"] = (
    100 * summary_by_provenance["trend_detected"] / summary_by_provenance["total"]
).round(2)
summary_by_provenance["detrended_percent"] = (
    100 * summary_by_provenance["detrended"] / summary_by_provenance["total"]
).round(2)

display(summary_by_provenance)

In [ ]:
if "family" in df.columns:
    summary_by_family = (
        df[df[provenanceColumn].isin(eligibleProvenance)]
        .groupby("family", dropna=False)
        .agg(
            total=("family", "size"),
            trend_detected=("trendDetectedForDetrend", "sum"),
            detrended=("detrended", "sum"),
            median_fraction_segments_with_drift=("trendFractionSegmentsWithDrift", "median"),
            median_drift_strength=("trendMedianSegmentDriftStrength", "median"),
            p75_drift_strength=("trendMedianSegmentDriftStrength", lambda x: x.dropna().quantile(0.75) if len(x.dropna()) else np.nan),
            p90_drift_strength=("trendMedianSegmentDriftStrength", lambda x: x.dropna().quantile(0.90) if len(x.dropna()) else np.nan),
        )
        .reset_index()
    )

    summary_by_family["trend_detected_percent"] = (
        100 * summary_by_family["trend_detected"] / summary_by_family["total"]
    ).round(2)
    summary_by_family["detrended_percent"] = (
        100 * summary_by_family["detrended"] / summary_by_family["total"]
    ).round(2)

    display(summary_by_family.sort_values("detrended_percent", ascending=False))
else:
    print("No family column found; skipping family summary.")

## 7. Save updated parquet and summaries

The output parquet keeps the original metadata and adds trend/detrending columns.

Suggested downstream usage:

```text
If detrended == True:
    use detrendLightCurvePath
else:
    use rawLightCurvePath or standardized light curve path depending on feature extraction design
```

This preserves scientific reproducibility because the original light curves are never overwritten.

In [ ]:
outputParquetFile = outputFolder / f"{inputParquetFile.stem}_tesscut_conditional_detrended.parquet"
df.to_parquet(outputParquetFile, index=False)

summaryProvenanceCsv = outputFolder / "tesscut_detrending_summary_by_provenance.csv"
summary_by_provenance.to_csv(summaryProvenanceCsv, index=False)

if "family" in df.columns:
    summaryFamilyCsv = outputFolder / "tesscut_detrending_summary_by_family.csv"
    summary_by_family.to_csv(summaryFamilyCsv, index=False)

print("Saved updated parquet:")
print(outputParquetFile)

print("Saved provenance summary:")
print(summaryProvenanceCsv)

if "family" in df.columns:
    print("Saved family summary:")
    print(summaryFamilyCsv)

## 8. Recommended conclusion

Use this notebook as a controlled preprocessing experiment, not as an irreversible pipeline step.

Recommended interpretation:

```text
TESSCut light curves are conditionally detrended only when segment-wise robust drift is detected.
SPOC and QLP light curves are not detrended in this workflow.
The detrending result is stored as an alternative light curve product, while the original raw light curve is preserved.
```

For ML, compare:

1. Baseline features from original light curves.
2. Features using conditionally detrended TESSCut light curves.
3. Optional stress test: all TESSCut detrended.

The second option should be your default candidate, but the comparison will tell whether detrending improves classification performance.